# 00 Train YOLO11n 8-Class Sign + Traffic Detector on Colab

이 노트북의 목적은 Roboflow에서 새로 export한 **8-class YOLOv11 dataset**을 Colab에서 바로 학습하고, 학습이 끝나면 **ONNX까지 한 번에 export**하는 것이다.

이번 버전의 핵심 차이:

- 기존 sign 6-class에 traffic light class를 추가한다.
- `left/right` 표지판이 있기 때문에 horizontal flip은 명시적으로 끈다.
- `red/green` traffic light는 색 의미가 중요하므로 YOLO 내부 HSV augmentation을 약하게 제한한다.
- Roboflow에서 이미 3x augmentation을 적용했으므로, Colab 학습 단계에서는 과한 추가 augmentation을 피한다.

최종 산출물:

```text
DRIVE_OUTPUT_DIR/
  sign_traffic_8class_yolo11n_v1/
    weights/best.pt
    weights/best.onnx
    results.csv
    confusion_matrix.png
  selected_export/
    sign_traffic_8class_yolo11n_v1_best.pt
    sign_traffic_8class_yolo11n_v1_best.onnx
    sign_traffic_8class_yolo11n_v1_classes.json
    sign_traffic_8class_yolo11n_v1_train_config.json
```

## 0. Colab Runtime 준비

권장 runtime:

```text
Runtime > Change runtime type > T4 GPU
```

처음 실행 시 `ultralytics`를 설치한다. Colab 기본 환경이 바뀌어도 재현 가능하도록 version과 GPU 정보를 출력한다.

In [ ]:
!pip -q install ultralytics pyyaml

from pathlib import Path
import os
import json
import shutil
import zipfile
from datetime import datetime

import yaml
import pandas as pd
import numpy as np
import torch
from ultralytics import YOLO

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

try:
    import ultralytics
    print('ultralytics:', ultralytics.__version__)
except Exception as e:
    print('ultralytics version check failed:', e)

## 1. Google Drive Mount와 경로 설정

아래 두 경로만 맞으면 된다.

- `DRIVE_ZIP_DIR`: Roboflow에서 받은 `.zip`을 올려둔 폴더
- `DRIVE_OUTPUT_DIR`: 학습 결과와 ONNX export 결과가 저장될 폴더

주의: 첫 번째 경로는 사용자가 지정한 `08_YOLO_sign_traffic_detection`이고, 두 번째 경로는 사용자가 지정한 `08_YOLO_sign_detection/runs_8class_yolo11n`을 그대로 사용한다.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

DRIVE_ZIP_DIR = Path('/content/drive/MyDrive/Colab Notebooks/26-1학기_임베디드인공지능시스템최적화/08_YOLO_sign_traffic_detection')
DRIVE_OUTPUT_DIR = Path('/content/drive/MyDrive/Colab Notebooks/26-1학기_임베디드인공지능시스템최적화/08_YOLO_sign_detection/runs_8class_yolo11n')

WORK_ROOT = Path('/content/sign_traffic_8class_yolo11n')
RAW_DATA_ROOT = WORK_ROOT / 'raw_roboflow_export'
REPORT_ROOT = WORK_ROOT / 'reports'
SELECTED_EXPORT_DIR = DRIVE_OUTPUT_DIR / 'selected_export'

for p in [DRIVE_ZIP_DIR, DRIVE_OUTPUT_DIR, WORK_ROOT, RAW_DATA_ROOT, REPORT_ROOT, SELECTED_EXPORT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('DRIVE_ZIP_DIR:', DRIVE_ZIP_DIR)
print('DRIVE_OUTPUT_DIR:', DRIVE_OUTPUT_DIR)
print('WORK_ROOT:', WORK_ROOT)

## 2. Roboflow ZIP 선택

이번에는 8-class dataset zip 하나만 사용한다.

ZIP 이름에 `sign_traffic`, `8class`, `yolov11` 등이 들어간 최신 파일을 우선 선택한다. 여러 개가 있으면 가장 최근 수정된 zip을 사용한다.

In [ ]:
ZIP_NAME_HINTS = ['sign_traffic', '8class', 'yolov11']

zip_files = sorted(DRIVE_ZIP_DIR.glob('*.zip'))
print('zip files:', len(zip_files))
for z in zip_files:
    print('-', z.name, f'{z.stat().st_size/1024/1024:.2f} MB')

assert zip_files, f'ZIP 파일을 찾지 못함: {DRIVE_ZIP_DIR}'

hinted = []
for z in zip_files:
    low = z.name.lower()
    score = sum(1 for h in ZIP_NAME_HINTS if h in low)
    hinted.append((score, z.stat().st_mtime, z))

_, _, ZIP_PATH = max(hinted, key=lambda x: (x[0], x[1]))
print('selected ZIP:', ZIP_PATH.name)
assert ZIP_PATH.exists(), ZIP_PATH

## 3. ZIP 압축 해제

Roboflow YOLO export는 일반적으로 아래 구조를 가진다.

```text
data.yaml
train/images, train/labels
valid/images, valid/labels
test/images, test/labels
```

이미 해제되어 있으면 재사용한다. 다시 풀고 싶으면 `FORCE_EXTRACT=True`로 바꾼다.

In [ ]:
FORCE_EXTRACT = False
DATASET_ROOT = RAW_DATA_ROOT / 'dataset'

if FORCE_EXTRACT and DATASET_ROOT.exists():
    shutil.rmtree(DATASET_ROOT)

if not (DATASET_ROOT / 'data.yaml').exists():
    if DATASET_ROOT.exists():
        shutil.rmtree(DATASET_ROOT)
    DATASET_ROOT.mkdir(parents=True, exist_ok=True)
    print('extracting:', ZIP_PATH.name, '->', DATASET_ROOT)
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(DATASET_ROOT)
else:
    print('reuse extracted dataset:', DATASET_ROOT)

DATA_YAML = DATASET_ROOT / 'data.yaml'
assert DATA_YAML.exists(), f'data.yaml이 없음: {DATA_YAML}'
print('DATA_YAML:', DATA_YAML)

## 4. data.yaml과 class 확인

여기서는 학습을 막을 정도의 검사는 최소화한다.

필수 확인:

- `nc == 8`
- class name 8개가 정상 출력되는지
- train/valid/test split 경로가 존재하는지

class name이 예상과 다르면 학습은 가능하지만, 나중에 이벤트 트리거 config에서 class 이름을 반드시 맞춰야 한다.

In [ ]:
def normalize_names(names_obj):
    if isinstance(names_obj, dict):
        return [names_obj[k] for k in sorted(names_obj, key=lambda x: int(x))]
    return list(names_obj)

with open(DATA_YAML, 'r', encoding='utf-8') as f:
    data_cfg = yaml.safe_load(f)

names = normalize_names(data_cfg.get('names', []))
nc = int(data_cfg.get('nc', len(names)))
print('nc:', nc)
print('names:', names)

EXPECTED_NAME_SET = {'horn', 'left', 'right', 'speed_20', 'stop', 'straight', 'traffic_red', 'traffic_green'}
actual_set = set(names)
print('missing from expected:', sorted(EXPECTED_NAME_SET - actual_set))
print('extra names:', sorted(actual_set - EXPECTED_NAME_SET))

assert nc == 8, f'8-class dataset이어야 함. 현재 nc={nc}, names={names}'
assert len(names) == 8, f'class name 8개가 필요함. 현재 {len(names)}개: {names}'

for split in ['train', 'valid', 'test']:
    img_dir = DATASET_ROOT / split / 'images'
    lab_dir = DATASET_ROOT / split / 'labels'
    print(split, 'images exists:', img_dir.exists(), 'labels exists:', lab_dir.exists())
    assert img_dir.exists(), img_dir
    assert lab_dir.exists(), lab_dir

class_info = {'nc': nc, 'names': names, 'data_yaml': str(DATA_YAML), 'zip_name': ZIP_PATH.name}
(REPORT_ROOT / 'class_info.json').write_text(json.dumps(class_info, ensure_ascii=False, indent=2), encoding='utf-8')

## 5. Dataset quick count

시간을 아끼기 위해 정밀 검수는 하지 않는다.

다만 split별 image/label 개수와 empty label 개수만 확인한다. `empty label`이 많으면 학습 자체는 가능하지만, Roboflow export나 annotation 상태를 다시 확인해야 한다.

In [ ]:
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

count_rows = []
for split in ['train', 'valid', 'test']:
    img_dir = DATASET_ROOT / split / 'images'
    lab_dir = DATASET_ROOT / split / 'labels'
    images = [p for p in img_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS]
    labels = sorted(lab_dir.glob('*.txt'))
    empty_labels = [p for p in labels if p.stat().st_size == 0]
    count_rows.append({
        'split': split,
        'images': len(images),
        'labels': len(labels),
        'empty_labels': len(empty_labels),
        'missing_label_estimate': max(0, len(images) - len(labels)),
    })

count_df = pd.DataFrame(count_rows)
display(count_df)
count_df.to_csv(REPORT_ROOT / 'dataset_quick_count.csv', index=False, encoding='utf-8-sig')

## 6. 학습 파라미터

8-class sign+traffic dataset에 맞춘 설정이다.

중요한 결정:

- `fliplr=0.0`: left/right 표지판 의미가 뒤집히면 안 되므로 horizontal flip 금지
- `hsv_h/hsv_s/hsv_v` 약하게 제한: red/green traffic class는 색 의미가 중요함
- `mosaic=0.0`: Roboflow에서 이미 3x augmentation을 적용했으므로, 첫 학습은 실제 카메라 이미지에 가까운 분포로 둠
- `epochs=80`: 기존 sign 모델과 같은 기준. 시간이 부족하면 그대로 두고, 부족하면 나중에 100~120 epoch로 재시도

In [ ]:
RUN_TRAINING = True
RUN_TEST_EVAL = True
RUN_PREDICT_PREVIEW = True
EXPORT_ONNX = True

MODEL_NAME = 'yolo11n.pt'
RUN_NAME = 'sign_traffic_8class_yolo11n_v1'
IMGSZ = 640
EPOCHS = 80
BATCH = 16
PATIENCE = 20
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
SEED = 0
WORKERS = 2

# 8-class 특성상 left/right와 red/green 의미가 보존되어야 하므로 augmentation을 제한한다.
AUG_KWARGS = dict(
    fliplr=0.0,
    flipud=0.0,
    hsv_h=0.01,
    hsv_s=0.25,
    hsv_v=0.20,
    degrees=0.0,
    translate=0.10,
    scale=0.30,
    shear=0.0,
    perspective=0.0,
    mosaic=0.0,
    mixup=0.0,
    copy_paste=0.0,
    close_mosaic=0,
)

TRAIN_CONFIG = {
    'model': MODEL_NAME,
    'run_name': RUN_NAME,
    'imgsz': IMGSZ,
    'epochs': EPOCHS,
    'batch': BATCH,
    'patience': PATIENCE,
    'device': str(DEVICE),
    'seed': SEED,
    'workers': WORKERS,
    'data_yaml': str(DATA_YAML),
    'output_dir': str(DRIVE_OUTPUT_DIR),
    'augmentation': AUG_KWARGS,
}
print(json.dumps(TRAIN_CONFIG, ensure_ascii=False, indent=2))
(REPORT_ROOT / 'train_config.json').write_text(json.dumps(TRAIN_CONFIG, ensure_ascii=False, indent=2), encoding='utf-8')
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(DRIVE_OUTPUT_DIR / f'{RUN_NAME}_train_config.json').write_text(json.dumps(TRAIN_CONFIG, ensure_ascii=False, indent=2), encoding='utf-8')

## 7. YOLO11n 학습

학습이 끝나면 `DRIVE_OUTPUT_DIR/RUN_NAME/weights/best.pt`가 생성된다.

Colab이 중간에 끊기더라도 Drive에 run directory가 남는다. 같은 이름으로 재실행하면 `exist_ok=True` 때문에 같은 폴더를 덮어쓸 수 있으므로, 새 실험을 분리하고 싶으면 `RUN_NAME`을 바꾸면 된다.

In [ ]:
train_record = {}
if RUN_TRAINING:
    model = YOLO(MODEL_NAME)
    results = model.train(
        data=str(DATA_YAML),
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        device=DEVICE,
        project=str(DRIVE_OUTPUT_DIR),
        name=RUN_NAME,
        exist_ok=True,
        seed=SEED,
        patience=PATIENCE,
        workers=WORKERS,
        cache=False,
        verbose=True,
        plots=True,
        **AUG_KWARGS,
    )
    run_dir = Path(results.save_dir)
    train_record = {
        'run_name': RUN_NAME,
        'run_dir': str(run_dir),
        'best_pt': str(run_dir / 'weights' / 'best.pt'),
        'last_pt': str(run_dir / 'weights' / 'last.pt'),
        'data_yaml': str(DATA_YAML),
    }
    print(json.dumps(train_record, ensure_ascii=False, indent=2))
else:
    run_dir = DRIVE_OUTPUT_DIR / RUN_NAME
    train_record = {
        'run_name': RUN_NAME,
        'run_dir': str(run_dir),
        'best_pt': str(run_dir / 'weights' / 'best.pt'),
        'last_pt': str(run_dir / 'weights' / 'last.pt'),
        'data_yaml': str(DATA_YAML),
    }
    print('RUN_TRAINING=False. Reusing:', run_dir)

BEST_PT = Path(train_record['best_pt'])
assert BEST_PT.exists(), f'best.pt가 없음: {BEST_PT}'
print('BEST_PT:', BEST_PT)

## 8. 학습 결과 요약

Ultralytics의 `results.csv`에서 best epoch 기준 metric을 요약한다.

이 표는 빠른 확인용이다. 최종적으로는 `confusion_matrix.png`, `PR_curve.png`, preview prediction도 같이 확인한다.

In [ ]:
def summarize_results_csv(run_dir: Path):
    csv_path = run_dir / 'results.csv'
    assert csv_path.exists(), csv_path
    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]
    map_col = 'metrics/mAP50(B)' if 'metrics/mAP50(B)' in df.columns else None
    best_idx = int(df[map_col].idxmax()) if map_col else int(len(df) - 1)
    best = df.iloc[best_idx].to_dict()
    last = df.iloc[-1].to_dict()
    wanted = [
        'epoch',
        'metrics/precision(B)',
        'metrics/recall(B)',
        'metrics/mAP50(B)',
        'metrics/mAP50-95(B)',
        'train/box_loss',
        'train/cls_loss',
        'val/box_loss',
        'val/cls_loss',
    ]
    out = {'run_name': RUN_NAME, 'run_dir': str(run_dir), 'best_epoch_by_mAP50': best_idx}
    for k in wanted:
        if k in best:
            out[f'best_{k}'] = best[k]
        if k in last:
            out[f'last_{k}'] = last[k]
    return out

RUN_DIR = Path(train_record['run_dir'])
summary = summarize_results_csv(RUN_DIR)
summary_df = pd.DataFrame([summary])
display(summary_df)
summary_df.to_csv(DRIVE_OUTPUT_DIR / f'{RUN_NAME}_train_summary.csv', index=False, encoding='utf-8-sig')
print('saved:', DRIVE_OUTPUT_DIR / f'{RUN_NAME}_train_summary.csv')

## 9. Test split 평가

학습 중 validation은 valid split 기준이다.

아래 셀은 `best.pt`를 test split으로 다시 평가한다. 시간이 매우 급하면 건너뛸 수 있지만, 학습이 잘 되었는지 최소 확인으로는 실행하는 것이 좋다.

In [ ]:
eval_record = {}
if RUN_TEST_EVAL:
    model = YOLO(str(BEST_PT))
    metrics = model.val(
        data=str(DATA_YAML),
        split='test',
        imgsz=IMGSZ,
        batch=BATCH,
        device=DEVICE,
        project=str(DRIVE_OUTPUT_DIR),
        name=f'test_eval_{RUN_NAME}',
        exist_ok=True,
        plots=True,
    )
    eval_record = {
        'run_name': RUN_NAME,
        'best_pt': str(BEST_PT),
        'test_map50': float(metrics.box.map50),
        'test_map50_95': float(metrics.box.map),
        'test_precision_mean': float(np.mean(metrics.box.p)) if getattr(metrics.box, 'p', None) is not None else None,
        'test_recall_mean': float(np.mean(metrics.box.r)) if getattr(metrics.box, 'r', None) is not None else None,
    }
    eval_df = pd.DataFrame([eval_record])
    display(eval_df)
    eval_df.to_csv(DRIVE_OUTPUT_DIR / f'{RUN_NAME}_test_eval_summary.csv', index=False, encoding='utf-8-sig')
else:
    print('RUN_TEST_EVAL=False. skipped.')

## 10. Preview prediction 저장

test image 일부에 대해 prediction image를 저장한다.

여기서는 정밀 검증이 아니라 빠르게 눈으로 확인하기 위한 preview다.

확인할 것:

- 기존 6개 sign class가 여전히 정상 탐지되는가
- traffic red/green bbox가 신호등 전체 또는 의미 있는 영역에 잘 잡히는가
- class가 red/green 사이에서 뒤바뀌지 않는가

In [ ]:
PREDICT_CONF = 0.25
PREDICT_SAMPLES = 64

if RUN_PREDICT_PREVIEW:
    test_images = sorted((DATASET_ROOT / 'test' / 'images').glob('*'))[:PREDICT_SAMPLES]
    print('preview sample images:', len(test_images))
    model = YOLO(str(BEST_PT))
    model.predict(
        source=[str(p) for p in test_images],
        imgsz=IMGSZ,
        conf=PREDICT_CONF,
        device=DEVICE,
        save=True,
        project=str(DRIVE_OUTPUT_DIR),
        name=f'predict_test_samples_{RUN_NAME}',
        exist_ok=True,
    )
else:
    print('RUN_PREDICT_PREVIEW=False. skipped.')

## 11. ONNX export

Pi runtime에서는 `.pt`가 아니라 ONNX를 사용하는 방향이다.

여기서는 학습된 `best.pt`를 바로 static batch-1 ONNX로 export한다.

Export 설정:

- `format='onnx'`
- `imgsz=640`
- `opset=12`
- `simplify=True`
- `dynamic=False`

`dynamic=False`인 이유는 Pi에서 입력 크기를 고정해 runtime overhead를 줄이고, 이전 sign runtime 방식과 맞추기 위함이다.

In [ ]:
export_record = {}
if EXPORT_ONNX:
    model = YOLO(str(BEST_PT))
    onnx_path = Path(model.export(
        format='onnx',
        imgsz=IMGSZ,
        opset=12,
        simplify=True,
        dynamic=False,
    ))
    assert onnx_path.exists(), onnx_path

    selected_pt = SELECTED_EXPORT_DIR / f'{RUN_NAME}_best.pt'
    selected_onnx = SELECTED_EXPORT_DIR / f'{RUN_NAME}_best.onnx'
    selected_classes = SELECTED_EXPORT_DIR / f'{RUN_NAME}_classes.json'
    selected_config = SELECTED_EXPORT_DIR / f'{RUN_NAME}_train_config.json'

    shutil.copy2(BEST_PT, selected_pt)
    shutil.copy2(onnx_path, selected_onnx)
    selected_classes.write_text(json.dumps(class_info, ensure_ascii=False, indent=2), encoding='utf-8')
    selected_config.write_text(json.dumps(TRAIN_CONFIG, ensure_ascii=False, indent=2), encoding='utf-8')

    export_record = {
        'run_name': RUN_NAME,
        'best_pt': str(BEST_PT),
        'onnx_path': str(onnx_path),
        'selected_pt': str(selected_pt),
        'selected_onnx': str(selected_onnx),
        'selected_classes': str(selected_classes),
        'selected_config': str(selected_config),
    }
    export_df = pd.DataFrame([export_record])
    display(export_df)
    export_df.to_csv(DRIVE_OUTPUT_DIR / f'{RUN_NAME}_onnx_export_summary.csv', index=False, encoding='utf-8-sig')
else:
    print('EXPORT_ONNX=False. skipped.')

## 12. 최종 산출물 체크리스트

학습이 끝난 뒤 Drive에서 아래 파일을 내려받으면 된다.

필수:

```text
DRIVE_OUTPUT_DIR/selected_export/sign_traffic_8class_yolo11n_v1_best.pt
DRIVE_OUTPUT_DIR/selected_export/sign_traffic_8class_yolo11n_v1_best.onnx
DRIVE_OUTPUT_DIR/selected_export/sign_traffic_8class_yolo11n_v1_classes.json
DRIVE_OUTPUT_DIR/sign_traffic_8class_yolo11n_v1_train_summary.csv
DRIVE_OUTPUT_DIR/sign_traffic_8class_yolo11n_v1_test_eval_summary.csv
```

시각 확인용:

```text
DRIVE_OUTPUT_DIR/sign_traffic_8class_yolo11n_v1/confusion_matrix.png
DRIVE_OUTPUT_DIR/sign_traffic_8class_yolo11n_v1/PR_curve.png
DRIVE_OUTPUT_DIR/predict_test_samples_sign_traffic_8class_yolo11n_v1/
```

다음 단계:

1. `best.pt / best.onnx / classes.json`을 로컬 shared model 폴더로 회수한다.
2. `20_sign_event_contract` 쪽 event trigger class list를 8-class 기준으로 업데이트한다.
3. Pi 패키지에서는 기존 sign ONNX를 이 8-class ONNX로 교체하고, traffic event를 YOLO event로 같이 처리한다.